# Replication probe: group-ablation flip rate (Qwen2.5-VL-3B)

The cheapest piece of the replication gate (plan section 4, step 2). It does
**no patching**: it takes Lietzow's published promoting heads from
`configs/heads_lietzow_qwen3b.json`, zero-ablates them at the last token, and
measures how many prior-grounded predictions flip.

Their reported numbers for Qwen-VL 3B: **prior flip 84.9%**, visual flip 1.4%,
on **73** correctly-conflicting examples of 467.

If this lands near 84.9%, the pipeline -- prompts, tokenisation, head indexing,
ablation site, quantisation -- is right, and the full patching sweep is worth
its GPU hours. If it does not, fix that first: the sweep cannot diagnose it.

Precision is **4-bit** (D10), matching their default.

**Section 5** (added 2026-10-27) re-runs the knockout with Lietzow's **pairwise**
flip definition (D67) -- group and single-head -- which is what D40 parts 1 and
3 are judged on. Sections 3-4 keep the earlier argmax definition for the record.

**Runtime.** About 1,100 forward passes (469 objects x 2 prompts, then ~75 x 2
ablated), roughly 1 h on one T4, plus ~15 min for section 5. Section 1b checks the speed before the
long loops start -- the first attempt ran at ~1 min per generation and died
after 16 h. Results are appended to `/kaggle/working/replication/*.jsonl` as
they come, so a kernel restart resumes instead of starting over. For an
unattended run use **Save Version -> Save & Run All**. Delete that folder if you
change a prompt or the head set, or it will serve stale answers.

Settings: GPU T4, Internet ON.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/SohamRShinde/VLM-Overlap-RP"
REPO_DIR = "/kaggle/working/research_paper"

import importlib, os, subprocess, sys

if REPO_URL:
    if os.path.exists(REPO_DIR):
        # Always update an existing checkout: a stale clone silently runs old
        # code, and the failure shows up later as a confusing ImportError.
        subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all", "-q"], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "-q", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)

if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

# Drop any already-imported package modules: Python caches them, so after an
# update the kernel would keep serving the old file and new names would appear
# to be missing.
for _m in [m for m in list(sys.modules) if m.startswith("circuit_overlap")]:
    del sys.modules[_m]
importlib.invalidate_caches()

COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("repo at", COMMIT)   # record this in RESULTS.md with any number from this run

# Fail here, loudly, if the checkout is older than this notebook expects.
# Otherwise a missing field surfaces much later as a confusing TypeError.
from dataclasses import fields as _fields
from circuit_overlap.notes import NoteSpec as _NoteSpec
from circuit_overlap.t_screen import ScreenConfig as _Cfg
import inspect as _inspect
from circuit_overlap.t_screen import run_screen as _run

_need = {
    "NoteSpec.mode / .uppercase": {"mode", "uppercase"} - {f.name for f in _fields(_NoteSpec)},
    "ScreenConfig.variants / .batch_size": {"variants", "batch_size"} - {f.name for f in _fields(_Cfg)},
    "run_screen(checkpoint_path=...)": {"checkpoint_path"} - set(_inspect.signature(_run).parameters),
}
import importlib.util as _ilu
for _mod in ("circuit_overlap.patching", "circuit_overlap.replication"):
    if _ilu.find_spec(_mod) is None:
        _need[_mod] = {"module"}
_missing = {k: sorted(v) for k, v in _need.items() if v}
if _missing:
    raise RuntimeError(
        f"checkout {COMMIT} is missing {_missing}. "
        "Push the local commits to origin/main, then re-run this cell."
    )
print("code is current for this notebook")

In [ ]:
# Use one GPU. On "GPU T4 x2", device_map="auto" splits the model across both
# cards, which only adds transfer time for a 3B model. This must run before
# torch is first imported, so restart the kernel if torch is already loaded.
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

# bitsandbytes is required for the 4-bit load (D10). transformers may cache
# "is bitsandbytes available?" at import time, so if the 4-bit load still
# fails right after installing, restart the kernel and run from cell 1.
!pip install -q -U "transformers>=4.49" accelerate datasets inflect "bitsandbytes>=0.46.1"
import torch, transformers
from transformers.utils import is_bitsandbytes_available
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))
from circuit_overlap.t_screen import native_bf16
# Not torch.cuda.is_bf16_supported(): it says True on a T4 because it counts
# slow emulation (CLAUDE.md gotchas).
print("bitsandbytes available:", is_bitsandbytes_available(),
      "| native bf16:", native_bf16(torch.cuda.get_device_capability(0)))

# Visual-Counterfact is read with an HF token (Lietzow's README).
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
from circuit_overlap.heads import load_head_set, format_head_label
hs = load_head_set(f"{REPO_DIR}/configs/heads_lietzow_qwen3b.json")
print(hs.name, hs.checkpoint)
print("promoting :", [format_head_label(h) for h in hs.promoting])
print("suppressing:", [format_head_label(h) for h in hs.suppressing])
print("layer range:", hs.layer_range(), " reported prior flip: 84.9%")

In [ ]:
from circuit_overlap.t_screen import ScreenConfig, VISUAL_PROMPT, PRIOR_PROMPT, load_model
config = ScreenConfig()
model, processor = load_model(config)        # 4-bit, nf4, double quant (D10)
print("devices used:", sorted({str(d) for d in getattr(model, "hf_device_map", {"": model.device}).values()}))

# [VERIFY] the head geometry before ablating anything: a mismatch here is the
# single most likely cause of a failed gate.
text_config = model.model.language_model.config
n_layers = text_config.num_hidden_layers
n_heads = text_config.num_attention_heads
head_dim = getattr(text_config, "head_dim", text_config.hidden_size // n_heads)
print(f"{n_layers} layers x {n_heads} heads, head_dim={head_dim}")
load_head_set(f"{REPO_DIR}/configs/heads_lietzow_qwen3b.json",
              n_layers=n_layers, n_heads_per_layer=n_heads)   # raises if 1-based

## 1b. [VERIFY] Image size and speed

Two things made the first run ~30x too slow, and both are cheap to rule out
here:

* **Image token budget.** `load_model` pins the image area with
  `min_pixels == max_pixels` (~1,280 image tokens, as Lietzow do). Newer
  transformers warns that those arguments are deprecated; if they are ignored,
  images go in at full resolution with many times more tokens. The cell
  rebuilds the processor with the newer `size=` argument if so.
* **Device.** The model must be on the GPU.

Do not run section 2 until the forward pass is a few seconds at most.

In [ ]:
import time, torch
from datasets import load_dataset
from transformers import AutoProcessor

ds = load_dataset("mgolov/Visual-Counterfact", split="color")
_probe = ds[0]["counterfact_image"].convert("RGB")

def build_inputs(image, prompt):
    messages = [{"role": "user",
                 "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return processor(text=[text], images=[image], return_tensors="pt").to(model.device)

def n_image_tokens(image):
    t, h, w = build_inputs(image, "x")["image_grid_thw"][0].tolist()
    return t * h * w // 4                    # 2x2 patch merge -> LLM tokens

EXPECTED = config.pixels // (28 * 28)        # 1280
n_img = n_image_tokens(_probe)
print(f"image {_probe.size} -> {n_img} image tokens (expected ~{EXPECTED})")
if n_img > 1.2 * EXPECTED:
    # min_pixels/max_pixels were ignored; the newer API takes them as `size`.
    processor = AutoProcessor.from_pretrained(
        config.checkpoint, use_fast=True,
        size={"shortest_edge": config.pixels, "longest_edge": config.pixels},
    )
    n_img = n_image_tokens(_probe)
    print(f"rebuilt the processor with size=... -> {n_img} image tokens")
assert n_img <= 1.2 * EXPECTED, "image token budget is still wrong -- do not run the loops"

assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4 and restart"
_inp = build_inputs(_probe, VISUAL_PROMPT.format(object="banana"))
for attempt in ("warm-up", "timed"):
    torch.cuda.synchronize(); t0 = time.time()
    with torch.no_grad():
        model(**_inp, use_cache=False)
    torch.cuda.synchronize()
    print(f"{attempt} forward pass: {time.time() - t0:.1f}s")
print("expect the timed pass to take ~0.5-3 s on a T4")

## 2. The conflicting subset

Per their README, the conflicting subset is the examples whose **top-1
prediction differs between the two prompts**, both on the **counterfactual**
image:

| Run | Prompt | Counts as right when top-1 is |
|---|---|---|
| Visual | `VISUAL_PROMPT` | the colour the image shows (`incorrect_answer`) |
| Prior | `PRIOR_PROMPT` | one of the real colours (`correct_answer`) |

"Correctly conflicting" (their 73) is both right at once. Top-1 is the argmax of
the **first answer token's** logits from one forward pass -- the position the
ablation acts on -- so no generation is needed. A colour's token ids follow
their convention (`color_first_token_ids`: both cases, first token, no leading
space).

The row filters are the ones `notebooks/t_screen.ipynb` cell 5 uses. They keep
**469** objects where Lietzow keep 467 (22 rows dropped for the name rule
against their implied 24) -- part of the open 493-vs-467 question (D20).
Record it next to the result.

In [ ]:
import inflect, json, pathlib, time
import torch
from collections import Counter
from circuit_overlap.data import parse_color_list, lietzow_object_name
from circuit_overlap.t_screen import color_first_token_ids

_eng = inflect.engine()
WORK = pathlib.Path("/kaggle/working/replication"); WORK.mkdir(parents=True, exist_ok=True)

# Same three row filters as t_screen cell 5. Images are read from `ds` when
# needed rather than held in memory.
rows, dropped = [], Counter()
for i, row in enumerate(ds):
    try:
        name = lietzow_object_name(row["object"], inflect_engine=_eng)
    except ValueError:
        dropped["name"] += 1; continue
    try:
        correct = parse_color_list(row["correct_answer"], require_list=True)
        attack = parse_color_list(row["incorrect_answer"])[0]
    except (ValueError, TypeError):
        dropped["colours"] += 1; continue
    if attack in correct:                    # radish and spider, per their README
        dropped["overlap"] += 1; continue
    rows.append({"i": i, "name": name, "correct": sorted(correct), "attack": attack})
print(len(rows), "objects", dict(dropped), "(Lietzow: 467)")

vocab = sorted({c for r in rows for c in r["correct"]} | {r["attack"] for r in rows})
color_ids, collisions = color_first_token_ids(processor.tokenizer, vocab)
print(len(vocab), "colours | sharing a first token (D26):", collisions)

def prompts(r):
    return (VISUAL_PROMPT.format(object=r["name"]),
            PRIOR_PROMPT.format(a_object=_eng.a(r["name"])))

def top1(image, prompt):
    with torch.no_grad():
        logits = model(**build_inputs(image, prompt), use_cache=False).logits[0, -1]
    return int(logits.argmax())

def both_top1(r):
    image = ds[r["i"]]["counterfact_image"].convert("RGB")
    visual_prompt, prior_prompt = prompts(r)
    return {"visual": top1(image, visual_prompt), "prior": top1(image, prior_prompt)}

def run_cached(path, todo, fn):
    """Apply fn to each row, appending to a JSONL file so a restart resumes."""
    done = {}
    if path.exists():
        for line in path.read_text().splitlines():
            d = json.loads(line); done[d["i"]] = d
    print(f"{path.name}: {sum(r['i'] in done for r in todo)} cached, "
          f"{sum(r['i'] not in done for r in todo)} to run", flush=True)
    t0, n_new = time.time(), 0
    with path.open("a") as f:
        for r in todo:
            if r["i"] in done:
                continue
            d = {"i": r["i"], **fn(r)}
            f.write(json.dumps(d) + "\n"); f.flush(); done[r["i"]] = d
            n_new += 1
            if n_new % 25 == 0:
                print(f"  {sum(x['i'] in done for x in todo)}/{len(todo)}  "
                      f"{(time.time() - t0) / n_new:.1f} s/object", flush=True)
    return [done[r["i"]] for r in todo]

base = run_cached(WORK / "baseline_top1.jsonl", rows, both_top1)

def ids_of(colors):
    return {t for c in colors for t in color_ids[c]}

# Objects whose colours share a first token with another colour are ambiguous
# under a first-token test, so they are left out of the subset.
ambiguous = set(collisions)
differs = [r for r, b in zip(rows, base) if b["visual"] != b["prior"]]
correctly = [
    r for r, b in zip(rows, base)
    if not ambiguous & ({r["attack"]} | set(r["correct"]))
    and b["visual"] in ids_of([r["attack"]])
    and b["prior"] in ids_of(r["correct"])
]
print("top-1 differs between the prompts:", len(differs))
print("correctly conflicting:", len(correctly), "of", len(rows), "(Lietzow: 73 of 467)")

# Eyeball a few: the decoded tokens should be colour words.
tok = processor.tokenizer
for r, b in list(zip(rows, base))[:8]:
    print(f"{r['name']:>14}  shows {r['attack']:<7} real {r['correct']}  "
          f"visual={tok.decode([b['visual']])!r}  prior={tok.decode([b['prior']])!r}")

## 3. Ablation

Lietzow zero the **o_proj input slice** `[head*head_dim : (head+1)*head_dim]`
at the **last token** (`src/vdpo/interventions.py::head_knockout`, MIT).
`head_knockout` below re-implements it with a forward pre-hook; diffed against
theirs at commit 145b453 on 2026-10-27, it zeroes the same slice at the same
position.

Flip rate = fraction of the correctly-conflicting examples whose top-1 changes
once all promoting heads are ablated; `circuit_overlap.metrics.flip_rate`
computes it. Only that subset is re-run, so this section is quick.

In [ ]:
from contextlib import contextmanager
from circuit_overlap.metrics import flip_rate

@contextmanager
def head_knockout(heads, head_dim):
    """Zero each head's slice of the o_proj input at the last token."""
    by_layer = {}
    for layer, head in heads:
        by_layer.setdefault(layer, []).append(head)
    layers = model.model.language_model.layers
    handles = []
    for layer, layer_heads in by_layer.items():
        def pre_hook(module, args, layer_heads=tuple(layer_heads)):
            x = args[0].clone()
            for h in layer_heads:
                x[:, -1, h * head_dim:(h + 1) * head_dim] = 0
            return (x, *args[1:])
        handles.append(layers[layer].self_attn.o_proj.register_forward_pre_hook(pre_hook))
    try:
        yield
    finally:
        for handle in handles:
            handle.remove()

# [VERIFY] an empty knockout changes nothing, the real one changes the logits.
r0 = correctly[0]
inp0 = build_inputs(ds[r0["i"]]["counterfact_image"].convert("RGB"), prompts(r0)[1])
with torch.no_grad():
    ref = model(**inp0, use_cache=False).logits[0, -1].float()
    with head_knockout([], head_dim):
        same = model(**inp0, use_cache=False).logits[0, -1].float()
    with head_knockout(hs.promoting, head_dim):
        knocked = model(**inp0, use_cache=False).logits[0, -1].float()
print("empty knockout     max |dlogit|:", (same - ref).abs().max().item(), "(expect ~0)")
print("promoting knockout max |dlogit|:", (knocked - ref).abs().max().item(), "(expect > 0)")

def both_top1_ablated(r):
    with head_knockout(hs.promoting, head_dim):
        return both_top1(r)

after = run_cached(WORK / "ablated_top1.jsonl", correctly, both_top1_ablated)
before = {b["i"]: b for b in base}
prior_before  = [before[r["i"]]["prior"]  for r in correctly]
visual_before = [before[r["i"]]["visual"] for r in correctly]
prior_after   = [a["prior"]  for a in after]
visual_after  = [a["visual"] for a in after]

prior = flip_rate(prior_before, prior_after)      # expect ~0.849
visual = flip_rate(visual_before, visual_after)   # expect ~0.014
print(f"prior flip  {prior:.1%}  (Lietzow 84.9%)   n={len(correctly)}")
print(f"visual flip {visual:.1%}  (Lietzow 1.4%)")

## 4. Record it

Write the result through `provenance.result_envelope` (model, checkpoint, n,
seed, prompt, commit) and add a row to `docs/RESULTS.md`. Note the compute
dtype: a T4 has no native bf16, so 4-bit compute runs in fp16 where Lietzow's
runs were probably bf16. The cell below writes
`/kaggle/working/replication/replication_flip_rate.json`; download it and copy
it into `results/`.

| Outcome | Reading |
|---|---|
| ~84.9% prior, ~1.4% visual | Pipeline verified; proceed to the patching sweep |
| Prior flip far lower | Check head indexing, the ablation site, the prompt suffix, and the conflicting-subset definition before anything else |
| Visual flip high | The ablation is hitting more than the intended heads |

In [ ]:
from circuit_overlap.provenance import result_envelope, write_result
from circuit_overlap.t_screen import native_bf16

envelope = result_envelope(
    "replication_flip_rate",
    {
        "prior_flip": prior,
        "visual_flip": visual,
        "n_correctly_conflicting": len(correctly),
        "n_top1_differs": len(differs),
        "n_objects": len(rows),
        "dropped": dict(dropped),
        "ambiguous_colours": sorted(ambiguous),
        "heads_ablated": [list(h) for h in hs.promoting],
        "reported": {"prior_flip": 0.849, "visual_flip": 0.014, "n": 73, "n_objects": 467},
    },
    model="Qwen2.5-VL-3B-Instruct",
    checkpoint=config.checkpoint,
    seed=config.seed,
    prompt=f"visual={VISUAL_PROMPT!r} prior={PRIOR_PROMPT!r}",
    n=len(correctly),
    repo_dir=REPO_DIR,
    quantisation="4-bit nf4, double quant",
    compute_dtype="bfloat16" if native_bf16(torch.cuda.get_device_capability(0)) else "float16",
    gpu=torch.cuda.get_device_name(0),
    image_tokens=n_img,
    prediction="argmax of first answer-token logits",
    ablation="zero o_proj input slice at last token (re-implemented, not their file)",
)
print(write_result(WORK / "replication_flip_rate.json", envelope))

## 5. Lietzow-definition knockout: group and single heads (D40 parts 1 and 3)

Section 3 counts a flip whenever the top-1 token changes. Lietzow's
`knockout_attn_heads` is **pairwise** (D67): after the knockout, a run counts as
prior-grounded when logit(prior run's answer) > logit(visual run's answer), and
a flip is a run that changes side. An answer moving to a third colour is not a
flip. This section re-runs their exact configurations -- each of the 15
classified heads alone, all 9 promoting heads together, all 6 suppressing
together -- on the same 73 objects, recording those two logits.

It uses the cached last-token path (`patching.LastTokenRunner`): the prompt
minus its last token is run once per object and only the last token is re-run
per knockout. The first cell checks that this equals a full forward pass, with
and without a knockout.

D40's thresholds (DECISIONS.md) were written on 2026-10-27, **before** this
section was run: (a) prior flip >= 2/3, (b) visual flip <= 10%, (f) L26H5 and
L31H3 in the top 3 single promoting heads. `scripts/analyze_replication_gate.py`
applies them to the saved file.

In [ ]:
from circuit_overlap.patching import LastTokenRunner
from circuit_overlap.replication import knockout_flip_counts, run_cached as run_jsonl

configs = {"grp_promoting": list(hs.promoting), "grp_suppressing": list(hs.suppressing)}
for h in list(hs.promoting) + list(hs.suppressing):
    configs[f"ind_{format_head_label(h)}"] = [h]

# [VERIFY] cached path == full forward pass, without and with a knockout. A
# runner must be built OUTSIDE any hook: its prefix run would otherwise zero
# the second-to-last token.
r0 = correctly[0]
runner0 = LastTokenRunner(model, build_inputs(ds[r0["i"]]["counterfact_image"].convert("RGB"),
                                              prompts(r0)[1]))
print("no knockout    max |dlogit| cached vs full:", runner0.check_against_full_forward(atol=1.0))
with head_knockout(hs.promoting, head_dim):
    print("group knockout max |dlogit| cached vs full:", runner0.check_against_full_forward(atol=1.0))
del runner0

def knockout_pairs(r):
    image = ds[r["i"]]["counterfact_image"].convert("RGB")
    visual_prompt, prior_prompt = prompts(r)
    a_prior, a_visual = before[r["i"]]["prior"], before[r["i"]]["visual"]
    runs = {"prior": LastTokenRunner(model, build_inputs(image, prior_prompt)),
            "visual": LastTokenRunner(model, build_inputs(image, visual_prompt))}
    out = {"a_prior": a_prior, "a_visual": a_visual}
    for name, heads in [("none", [])] + list(configs.items()):
        for kind, run in runs.items():
            with head_knockout(heads, head_dim):
                logits = run.logits()
            out[f"{name}/{kind}"] = [round(float(logits[a_prior]), 4),
                                     round(float(logits[a_visual]), 4)]
    return out

knock = run_jsonl(WORK / "knockout_pairwise.jsonl", correctly, knockout_pairs)
none = knockout_flip_counts(knock, "none")
print("sanity, no knockout -- expect 0 and 0 flips:", none["prior_flips"], none["visual_flips"])

In [ ]:
from circuit_overlap.provenance import result_envelope, write_result
from circuit_overlap.t_screen import native_bf16

counts = {name: knockout_flip_counts(knock, name) for name in ["none", *configs]}
for name in ("grp_promoting", "grp_suppressing"):
    c = counts[name]
    print(f"{name:16s} prior flip {c['prior_flips']:2d}/{c['n']} = {c['prior_flip_rate']:6.1%}   "
          f"visual flip {c['visual_flips']:2d}/{c['n']} = {c['visual_flip_rate']:6.1%}")
print("Lietzow, grp_promoting: prior 84.9%, visual 1.4%\n")

print("single promoting heads, by prior flip:")
singles = sorted(hs.promoting, key=lambda h: -counts[f"ind_{format_head_label(h)}"]["prior_flip_rate"])
for rank, h in enumerate(singles, 1):
    label = format_head_label(h)
    ref = hs.single_head_prior_flip_pct.get(label)
    print(f"  {rank}. {label:7s} {counts[f'ind_{label}']['prior_flip_rate']:6.1%}"
          + (f"   (Lietzow {ref}%)" if ref is not None else ""))

envelope = result_envelope(
    "replication_knockout_pairwise",
    {
        "flip_definition": "pairwise, Lietzow knockout_attn_heads (D67)",
        "configs": {k: [list(h) for h in v] for k, v in configs.items()},
        "counts": counts,
        "records": knock,
        "object_names": {r["i"]: r["name"] for r in correctly},
        "reported": {"grp_promoting": {"prior_flip": 0.849, "visual_flip": 0.014},
                     "single_head_prior_flip_pct": hs.single_head_prior_flip_pct, "n": 73},
    },
    model="Qwen2.5-VL-3B-Instruct",
    checkpoint=config.checkpoint,
    seed=config.seed,
    prompt=f"visual={VISUAL_PROMPT!r} prior={PRIOR_PROMPT!r}",
    n=len(correctly),
    repo_dir=REPO_DIR,
    quantisation="4-bit nf4, double quant",
    compute_dtype="bfloat16" if native_bf16(torch.cuda.get_device_capability(0)) else "float16",
    gpu=torch.cuda.get_device_name(0),
    image_tokens=n_img,
    ablation="zero o_proj input slice at last token; cached last-token path",
)
print("\n", write_result(WORK / "replication_knockout_qwen25vl3b.json", envelope))
print("Download it and put it in results/; then run scripts/analyze_replication_gate.py.")